In [ ]:
"""
Numerical example: relaxing a categorical distribution into the simplex.

Sequence: "cat" over vocabulary {c, a, t, o}  (K = 4)
"""

import numpy as np

np.set_printoptions(precision=3, suppress=True)

# ---------------------------------------------------------------
# Step 0: define vocabulary
# ---------------------------------------------------------------
vocab = ['c', 'a', 't', 'o']
K = len(vocab)
char_to_idx = {ch: i for i, ch in enumerate(vocab)}

print("=" * 60)
print("STEP 0 — Vocabulary")
print("=" * 60)
print(f"vocab     = {vocab}")
print(f"K         = {K}")
print(f"char->idx = {char_to_idx}")

# ---------------------------------------------------------------
# Step 1: tokenize the sequence "cat"
# ---------------------------------------------------------------
sequence = "cat"
indices = [char_to_idx[ch] for ch in sequence]

print()
print("=" * 60)
print("STEP 1 — Tokenize")
print("=" * 60)
print(f"sequence = {sequence!r}")
print(f"indices  = {indices}    (these are integers in {{0,...,{K-1}}})")

# ---------------------------------------------------------------
# Step 2: one-hot encode -> this IS x
# ---------------------------------------------------------------
# Standard basis vectors e_0, e_1, e_2, e_3 of R^4
I = np.eye(K)
print()
print("=" * 60)
print("STEP 2 — One-hot encode (this gives us x)")
print("=" * 60)
print("Standard basis vectors of R^K (the simplex vertices e_i):")
for i in range(K):
    print(f"  e_{i} ('{vocab[i]}') = {I[i]}")

x = np.stack([I[i] for i in indices])  # shape (3, K)
print()
print("x = sequence of one-hot vectors, one per token:")
for t, (ch, vec) in enumerate(zip(sequence, x)):
    print(f"  position {t} ('{ch}'): {vec}")
print(f"x.shape = {x.shape}    (3 tokens, each a vector in R^{K})")

# ---------------------------------------------------------------
# Step 3: describe the per-position distribution
# ---------------------------------------------------------------
# For a SINGLE deterministic token "a" (position 1), the distribution is
# just a delta at e_1. To make the mixture-of-deltas form non-trivial,
# pretend position 1 has empirical probabilities over the vocab.
print()
print("=" * 60)
print("STEP 3 — Describe distribution at one position as delta mixture")
print("=" * 60)
p = np.array([0.1, 0.7, 0.0, 0.2])
print(f"Suppose position 1 has categorical probs p = {p}")
print(f"  (sum = {p.sum()})")
print()
print("p_data(x) = sum_i p_i * delta(x - e_i)")
print("         =", " + ".join(
    f"{p[i]}*delta(x - e_{i})" for i in range(K) if p[i] > 0
))
print()
print("Sampling from this distribution returns one of these vectors:")
for i in range(K):
    if p[i] > 0:
        print(f"  e_{i} = {I[i]}   with probability {p[i]}")

# ---------------------------------------------------------------
# Step 4: continuous-space operation — add Gaussian noise
# ---------------------------------------------------------------
print()
print("=" * 60)
print("STEP 4 — Why the continuous view matters: add Gaussian noise")
print("=" * 60)

rng = np.random.default_rng(seed=0)
sigma = 0.1

# Take the clean "a" token = e_1
x_clean = I[1].copy()
print(f"Clean one-hot for 'a': x = {x_clean}")
print(f"Noise scale sigma = {sigma}")

eps = rng.standard_normal(K)
x_noisy = x_clean + sigma * eps
print(f"epsilon ~ N(0, I) sample: {eps}")
print(f"x_noisy = e_1 + sigma * epsilon =")
print(f"          {x_noisy}")

print()
print("Sanity checks on x_noisy:")
print(f"  sums to 1?       {np.isclose(x_noisy.sum(), 1.0)}  (sum = {x_noisy.sum():.4f})")
print(f"  all nonneg?      {np.all(x_noisy >= 0)}  (min = {x_noisy.min():.4f})")
print(f"  is a one-hot?    {np.any(np.allclose(x_noisy, I, atol=1e-6))}")
print("  => x_noisy left the simplex; it's a generic point in R^K.")
print("     But its 2nd entry is still ~1, so it 'points toward' e_1.")

# ---------------------------------------------------------------
# Summary table
# ---------------------------------------------------------------
print()
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"{'stage':<22}{'object':<32}{'lives in'}")
print("-" * 70)
print(f"{'raw text':<22}{repr(sequence):<32}strings")
print(f"{'token indices':<22}{str(indices):<32}{{0,...,{K-1}}}^3")
print(f"{'one-hot (= x)':<22}{'(e_0, e_1, e_2)':<32}vertices of S_{K} in R^{K}")
print(f"{'delta-mixture form':<22}{'sum_i p_i delta(x - e_i)':<32}measure on R^{K}")
print(f"{'noised':<22}{'e_i + sigma*epsilon':<32}generic point in R^{K}")

STEP 0 — Vocabulary
vocab     = ['c', 'a', 't', 'o']
K         = 4
char->idx = {'c': 0, 'a': 1, 't': 2, 'o': 3}

STEP 1 — Tokenize
sequence = 'cat'
indices  = [0, 1, 2]    (these are integers in {0,...,3})

STEP 2 — One-hot encode (this gives us x)
Standard basis vectors of R^K (the simplex vertices e_i):
  e_0 ('c') = [1. 0. 0. 0.]
  e_1 ('a') = [0. 1. 0. 0.]
  e_2 ('t') = [0. 0. 1. 0.]
  e_3 ('o') = [0. 0. 0. 1.]

x = sequence of one-hot vectors, one per token:
  position 0 ('c'): [1. 0. 0. 0.]
  position 1 ('a'): [0. 1. 0. 0.]
  position 2 ('t'): [0. 0. 1. 0.]
x.shape = (3, 4)    (3 tokens, each a vector in R^4)

STEP 3 — Describe distribution at one position as delta mixture
Suppose position 1 has categorical probs p = [0.1 0.7 0.  0.2]
  (sum = 1.0)

p_data(x) = sum_i p_i * delta(x - e_i)
         = 0.1*delta(x - e_0) + 0.7*delta(x - e_1) + 0.2*delta(x - e_3)

Sampling from this distribution returns one of these vectors:
  e_0 = [1. 0. 0. 0.]   with probability 0.1
  e_1 = [0. 1. 

In [15]:
import numpy as np
from scipy.stats import dirichlet

import matplotlib.pyplot as plt
plt.style.use("ggplot")
import seaborn as sns

alpha_base = np.ones(27) * 0.1
alpha_peak = np.zeros(27)
alpha_peak[26] = 10.0

dist = dirichlet.rvs(alpha=alpha_base+alpha_peak)

print(dist)

[[9.24869438e-03 1.62340763e-09 8.20899244e-05 1.24875958e-07
  1.74575121e-03 3.65367525e-10 5.44117925e-04 2.23969775e-05
  9.92449229e-02 1.36187434e-09 4.21693158e-09 5.55073871e-12
  1.00716041e-05 3.99747170e-02 3.42069039e-03 4.48360498e-03
  1.06199138e-07 4.35881517e-04 9.69140451e-03 7.29276284e-05
  1.63007172e-07 9.44088496e-11 8.49554733e-05 3.16656355e-09
  1.14497012e-01 2.81704406e-20 7.16440356e-01]]


In [3]:
"""
EqM + S-FLM Design A: time-free hyperspherical flow language model.

This is a minimal test harness for the proposal we sketched:
  - Tokens have unit-norm embeddings on S^{d-1} (a learned codebook).
  - A denoiser p_theta(v | z) is trained with cross-entropy on SLERP-noised
    latents, WITHOUT time conditioning.
  - The implicit energy is E_theta(z) = -tau * logsumexp(<h(z), e_v>/tau).
  - Sampling is Riemannian gradient descent on E_theta with adaptive steps,
    optionally with temperature annealing.

Task: a toy sequence-modeling problem (modular arithmetic) with a small
vocabulary, designed to fit on a single GPU in a couple of minutes.

Diagnostics:
  1. Does the energy actually place its minima at codeword positions?
  2. Does the Riemannian gradient field collapse far from codewords?
  3. Does sampling from uniform noise converge to valid sequences?

We compare three samplers on the same trained model:
  (A) Constant step gradient descent.
  (B) Adaptive step size (normalized by gradient norm).
  (C) Temperature annealing (high tau -> low tau).
"""

from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


# ---------------------------------------------------------------------------
# Spherical primitives
# ---------------------------------------------------------------------------


def normalize(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Project onto S^{d-1} along the last dim."""
    return x / x.norm(dim=-1, keepdim=True).clamp(min=eps)


def sample_uniform_sphere(shape: tuple[int, ...], device, dtype=torch.float32) -> torch.Tensor:
    """Uniform on S^{d-1}: Gaussian then normalize."""
    eps = torch.randn(shape, device=device, dtype=dtype)
    return normalize(eps)


def geodesic_distance(p: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    """Arc length on S^{d-1} between unit vectors p and q, along last dim."""
    dot = (p * q).sum(dim=-1).clamp(-1.0 + 1e-7, 1.0 - 1e-7)
    return torch.arccos(dot)


def slerp(p: torch.Tensor, q: torch.Tensor, alpha: torch.Tensor) -> torch.Tensor:
    """
    SLERP from p (at alpha=0) to q (at alpha=1), along the last axis.

    p, q: unit vectors with the same trailing dim.
    alpha: scalar or broadcastable to p's leading dims (no last dim).
    """
    omega = geodesic_distance(p, q).unsqueeze(-1)  # (..., 1)
    sin_omega = torch.sin(omega).clamp(min=1e-7)
    a = alpha.unsqueeze(-1) if alpha.dim() == p.dim() - 1 else alpha
    coef_p = torch.sin((1.0 - a) * omega) / sin_omega
    coef_q = torch.sin(a * omega) / sin_omega
    return coef_p * p + coef_q * q


def log_map(p: torch.Tensor, q: torch.Tensor) -> torch.Tensor:
    """
    log_p(q): tangent vector at p pointing toward q, with norm = geodesic distance.

    Returns zero when p == q (handled via clamping omega).
    """
    omega = geodesic_distance(p, q).unsqueeze(-1)
    sin_omega = torch.sin(omega).clamp(min=1e-7)
    return (omega / sin_omega) * (q - torch.cos(omega) * p)


def exp_map(p: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    exp_p(v): walk along the geodesic from p in direction v for arc length ||v||.
    """
    v_norm = v.norm(dim=-1, keepdim=True).clamp(min=1e-7)
    return torch.cos(v_norm) * p + torch.sin(v_norm) * (v / v_norm)


def project_tangent(p: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Project ambient vector v onto T_p S^{d-1}."""
    return v - (p * v).sum(dim=-1, keepdim=True) * p


# ---------------------------------------------------------------------------
# Toy task: modular-arithmetic sequences.
# Goal: model sequences "a + b = c" mod P, where c = (a + b) mod P.
# Vocabulary: digits 0..P-1, plus "+" and "=".
# This is small but nontrivial; an n-gram model can't solve it, the model
# must learn the addition structure to predict the answer position.
# ---------------------------------------------------------------------------


@dataclass
class ToyConfig:
    P: int = 7  # modulus
    seq_len: int = 5  # "a + b = c" tokenized as [a, PLUS, b, EQ, c]
    embed_dim: int = 64
    hidden_dim: int = 128
    n_layers: int = 2
    n_heads: int = 4
    batch_size: int = 256
    n_steps: int = 8000
    lr: float = 3e-4
    tau: float = 0.1  # temperature for the codebook softmax / energy
    alpha_lo: float = 0.0  # noise schedule lower bound (training samples in [lo, hi])
    alpha_hi: float = 0.95  # truncation bound (S-FLM eq. 16 analogue)
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


class ModularArithmeticDataset(Dataset):
    def __init__(self, cfg: ToyConfig, n_examples: int):
        self.cfg = cfg
        P = cfg.P
        # Vocab: 0..P-1 are digits, P is '+', P+1 is '='
        self.vocab_size = P + 2
        self.PLUS = P
        self.EQ = P + 1
        rng = torch.Generator().manual_seed(0)
        a = torch.randint(0, P, (n_examples,), generator=rng)
        b = torch.randint(0, P, (n_examples,), generator=rng)
        c = (a + b) % P
        plus = torch.full_like(a, self.PLUS)
        eq = torch.full_like(a, self.EQ)
        self.data = torch.stack([a, plus, b, eq, c], dim=1)  # (N, 5)

    def __len__(self) -> int:
        return self.data.shape[0]

    def __getitem__(self, idx: int) -> torch.Tensor:
        return self.data[idx]


# ---------------------------------------------------------------------------
# Model: a tiny transformer denoiser that maps a sequence of points on S^{d-1}
# to per-position logits over the vocabulary. NO time conditioning.
# ---------------------------------------------------------------------------


class TinyDenoiser(nn.Module):
    """
    Time-free denoiser. Maps z in (S^{d-1})^L to logits in R^{L x V}.

    Logits are computed via a learned feature h(z) dotted against the codebook:
        logits_v(z) = <h_theta(z), e_v> / tau
    so that the implicit energy at position ell is:
        E_theta(z^ell) = -tau * logsumexp_v <h_theta(z^ell), e_v> / tau.
    """

    def __init__(self, cfg: ToyConfig, vocab_size: int):
        super().__init__()
        self.cfg = cfg
        self.vocab_size = vocab_size
        # Learned codebook (NOT renormalized after each step; we project in
        # forward instead — see S-FLM Suppl. B.5 for why this is preferable).
        self.codebook = nn.Parameter(torch.randn(vocab_size, cfg.embed_dim) * 0.1)
        # Positional embeddings (additive in feature space, not on the sphere).
        self.pos_embed = nn.Parameter(torch.zeros(cfg.seq_len, cfg.hidden_dim))
        nn.init.normal_(self.pos_embed, std=0.02)
        # Encoder from sphere -> hidden
        self.in_proj = nn.Linear(cfg.embed_dim, cfg.hidden_dim)
        # Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.hidden_dim,
            nhead=cfg.n_heads,
            dim_feedforward=4 * cfg.hidden_dim,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.n_layers)
        # Decoder back to embed_dim (so we can dot with codebook)
        self.out_proj = nn.Linear(cfg.hidden_dim, cfg.embed_dim)

    def codebook_normalized(self) -> torch.Tensor:
        return normalize(self.codebook)

    def features(self, z: torch.Tensor) -> torch.Tensor:
        """
        z: (B, L, d) with each z[b, ell] on S^{d-1}.
        Returns h(z): (B, L, d) — the feature dotted against codebook for logits.
        """
        h = self.in_proj(z) + self.pos_embed.unsqueeze(0)
        h = self.transformer(h)
        h = self.out_proj(h)
        return h

    def logits(self, z: torch.Tensor) -> torch.Tensor:
        """Returns (B, L, V) logits — h(z) . e_v / tau."""
        h = self.features(z)  # (B, L, d)
        e_hat = self.codebook_normalized()  # (V, d)
        return (h @ e_hat.T) / self.cfg.tau

    def log_posterior(self, z: torch.Tensor) -> torch.Tensor:
        """Per-position log p_theta(v | z), shape (B, L, V)."""
        return F.log_softmax(self.logits(z), dim=-1)

    def energy(self, z: torch.Tensor, tau: float | None = None) -> torch.Tensor:
        """
        Per-position energy E_theta(z^ell) = -tau * logsumexp_v (h . e_v / tau).
        Returns (B, L).

        tau argument lets us anneal the energy at sampling time without retraining.
        """
        h = self.features(z)
        e_hat = self.codebook_normalized()
        eff_tau = self.cfg.tau if tau is None else tau
        scores = (h @ e_hat.T) / eff_tau
        return -eff_tau * torch.logsumexp(scores, dim=-1)

    def total_energy(self, z: torch.Tensor, tau: float | None = None) -> torch.Tensor:
        """Sum of per-position energies — a scalar per batch element."""
        return self.energy(z, tau=tau).sum(dim=-1)


# ---------------------------------------------------------------------------
# Training: cross-entropy on SLERP-noised latents, no time conditioning.
# ---------------------------------------------------------------------------


def encode_tokens(tokens: torch.Tensor, model: TinyDenoiser) -> torch.Tensor:
    """tokens: (B, L) long -> z1: (B, L, d) unit-norm."""
    e_hat = model.codebook_normalized()
    return e_hat[tokens]


def make_noisy_batch(
    z1: torch.Tensor, cfg: ToyConfig
) -> tuple[torch.Tensor, torch.Tensor]:
    """SLERP a uniform-noise sample toward z1 at a random alpha in [alpha_lo, alpha_hi]."""
    B, L, d = z1.shape
    z0 = sample_uniform_sphere((B, L, d), device=z1.device, dtype=z1.dtype)
    # Same alpha for all positions in a sequence (could vary; sequence-wise is simpler).
    alpha = torch.empty(B, 1, device=z1.device).uniform_(cfg.alpha_lo, cfg.alpha_hi)
    alpha_full = alpha.expand(B, L)
    z_alpha = slerp(z0, z1, alpha_full)
    return z_alpha, alpha.squeeze(-1)


def train(cfg: ToyConfig) -> tuple[TinyDenoiser, ModularArithmeticDataset]:
    dataset = ModularArithmeticDataset(cfg, n_examples=8192)
    loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True, drop_last=True)

    model = TinyDenoiser(cfg, vocab_size=dataset.vocab_size).to(cfg.device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    step = 0
    model.train()
    print(f"Training on {cfg.device} | vocab={dataset.vocab_size} | d={cfg.embed_dim}")
    while step < cfg.n_steps:
        for tokens in loader:
            tokens = tokens.to(cfg.device)
            z1 = encode_tokens(tokens, model)
            z_alpha, _ = make_noisy_batch(z1, cfg)
            log_p = model.log_posterior(z_alpha)  # (B, L, V)
            loss = F.nll_loss(
                log_p.reshape(-1, log_p.shape[-1]),
                tokens.reshape(-1),
            )
            opt.zero_grad()
            loss.backward()
            opt.step()
            step += 1
            if step % 500 == 0:
                # Quick accuracy check at low noise to sanity check learning.
                with torch.no_grad():
                    z_low = slerp(
                        sample_uniform_sphere(z1.shape, cfg.device),
                        z1,
                        torch.full((z1.shape[0], z1.shape[1]), 0.9, device=cfg.device),
                    )
                    preds = model.logits(z_low).argmax(-1)
                    acc = (preds == tokens).float().mean().item()
                print(f"  step {step:5d}  loss={loss.item():.4f}  low-noise acc={acc:.3f}")
            if step >= cfg.n_steps:
                break

    return model, dataset


# ---------------------------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------------------------


@torch.no_grad()
def diagnostic_energy_at_codewords(model: TinyDenoiser, dataset: ModularArithmeticDataset, cfg: ToyConfig) -> None:
    """
    Check whether the energy is genuinely minimized at codeword embeddings,
    vs. at the centroid (which would be the EqM-on-text failure mode).
    """
    print("\n=== Diagnostic 1: energy landscape minima ===")
    e_hat = model.codebook_normalized()  # (V, d)

    # Energy at each codeword (broadcast to a length-1 sequence).
    z_codes = e_hat.unsqueeze(1)  # (V, 1, d)
    # We need to plug into a length-L sequence; replicate the codeword across positions.
    z_codes_seq = e_hat.unsqueeze(1).expand(-1, cfg.seq_len, -1)  # (V, L, d)
    E_codes = model.energy(z_codes_seq).mean(dim=-1)  # (V,) average over positions

    # Energy at the centroid (uniform mean of codewords, normalized).
    centroid = normalize(e_hat.mean(dim=0))  # (d,)
    z_centroid = centroid.expand(1, cfg.seq_len, -1)
    E_centroid = model.energy(z_centroid).mean().item()

    # Energy at random points on the sphere.
    z_random = sample_uniform_sphere((128, cfg.seq_len, cfg.embed_dim), cfg.device)
    E_random = model.energy(z_random).mean().item()

    print(f"  Mean energy at codewords:    {E_codes.mean().item(): .4f}  (min: {E_codes.min().item():.4f})")
    print(f"  Energy at centroid:          {E_centroid: .4f}")
    print(f"  Mean energy at random init:  {E_random: .4f}")
    print(f"  Codeword < Centroid:  {E_codes.mean().item() < E_centroid}  (want True)")
    print(f"  Codeword < Random:    {E_codes.mean().item() < E_random}    (want True)")


@torch.enable_grad()
def diagnostic_gradient_magnitude(model: TinyDenoiser, cfg: ToyConfig) -> None:
    """
    Plot |grad E| as a function of distance to nearest codeword.
    A field that collapses far from codewords is the predicted failure mode.
    """
    print("\n=== Diagnostic 2: gradient field magnitude vs. distance to codebook ===")
    e_hat = model.codebook_normalized()  # (V, d)

    # Sample SLERP-noised points at various alpha levels.
    alphas = torch.linspace(0.05, 0.99, 10, device=cfg.device)
    results = []
    for alpha in alphas:
        # Build a batch of points at this alpha from random target codewords.
        B = 64
        target_ids = torch.randint(0, e_hat.shape[0], (B, cfg.seq_len), device=cfg.device)
        z1 = e_hat[target_ids]
        z0 = sample_uniform_sphere(z1.shape, cfg.device)
        z = slerp(z0, z1, alpha.expand(B, cfg.seq_len)).requires_grad_(True)

        E = model.total_energy(z).sum()
        grad, = torch.autograd.grad(E, z)
        grad_tan = project_tangent(z.detach(), grad)
        grad_norm = grad_tan.norm(dim=-1).mean().item()

        # Distance to nearest codeword for each position.
        with torch.no_grad():
            sims = z.detach() @ e_hat.T  # (B, L, V)
            nearest_sim = sims.max(dim=-1).values
            nearest_dist = torch.arccos(nearest_sim.clamp(-1 + 1e-7, 1 - 1e-7)).mean().item()
        results.append((alpha.item(), nearest_dist, grad_norm))

    print(f"  {'alpha':>6}  {'dist-to-nearest':>16}  {'|grad E|':>10}")
    for a, d, g in results:
        print(f"  {a:6.2f}  {d:16.4f}  {g:10.4f}")

    # Field-collapse check: gradient norm at high noise (alpha ~ 0.05) should be
    # nonzero. Compare to gradient norm near a codeword (alpha ~ 0.99).
    g_far = results[0][2]
    g_near = results[-1][2]
    print(f"  |grad| far from data:  {g_far:.4f}")
    print(f"  |grad| near data:      {g_near:.4f}")
    print(f"  Ratio near/far:        {g_near / max(g_far, 1e-8):.2f}")
    print(
        "  (If ratio >> 1, field collapses far from data — sampler will struggle.)"
    )


# ---------------------------------------------------------------------------
# Sampling: Riemannian gradient descent on E_theta with three step strategies.
# ---------------------------------------------------------------------------


def riemannian_grad(model: TinyDenoiser, z: torch.Tensor, tau: float | None = None) -> torch.Tensor:
    """Compute projected gradient of total energy w.r.t. z."""
    z = z.detach().requires_grad_(True)
    E = model.total_energy(z, tau=tau).sum()
    grad, = torch.autograd.grad(E, z)
    return project_tangent(z.detach(), grad)


def sample_constant_step(model, cfg, n_seqs: int, n_steps: int, eta: float):
    z = sample_uniform_sphere((n_seqs, cfg.seq_len, cfg.embed_dim), cfg.device)
    for _ in range(n_steps):
        g = riemannian_grad(model, z)
        z = exp_map(z, -eta * g)
        z = normalize(z)  # numerical safety
    return z


def sample_adaptive_step(model, cfg, n_seqs: int, n_steps: int, target_step: float = 0.1):
    """Normalize step by gradient magnitude so we move ~target_step per iteration."""
    z = sample_uniform_sphere((n_seqs, cfg.seq_len, cfg.embed_dim), cfg.device)
    for _ in range(n_steps):
        g = riemannian_grad(model, z)
        g_norm = g.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        # Cap the step at target_step in arc length per position.
        eta = (target_step / g_norm).clamp(max=1.0)
        z = exp_map(z, -eta * g)
        z = normalize(z)
    return z


def sample_temperature_anneal(model, cfg, n_seqs: int, n_steps: int, tau_hi: float = 1.0, tau_lo: float = 0.05, eta: float = 0.1):
    """Anneal energy temperature from high (smooth) to low (peaked)."""
    z = sample_uniform_sphere((n_seqs, cfg.seq_len, cfg.embed_dim), cfg.device)
    for k in range(n_steps):
        # Linear in log-space.
        frac = k / max(n_steps - 1, 1)
        tau = math.exp(math.log(tau_hi) * (1 - frac) + math.log(tau_lo) * frac)
        g = riemannian_grad(model, z, tau=tau)
        g_norm = g.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        # Adaptive step inside the anneal, since |grad| changes a lot with tau.
        step = (0.1 / g_norm).clamp(max=1.0)
        z = exp_map(z, -step * g)
        z = normalize(z)
    return z


# ---------------------------------------------------------------------------
# Eval: decode latents to tokens, check arithmetic correctness.
# ---------------------------------------------------------------------------


@torch.no_grad()
def decode_and_score(model, dataset, cfg, z):
    """Decode latents with arg max and count correct sequences."""
    logits = model.logits(z)  # (B, L, V)
    preds = logits.argmax(-1)  # (B, L)
    # A sequence is "valid" if positions 1, 3 are PLUS, EQ, and a + b == c mod P.
    plus_ok = preds[:, 1] == dataset.PLUS
    eq_ok = preds[:, 3] == dataset.EQ
    digits_ok = (preds[:, [0, 2, 4]] < dataset.cfg.P).all(dim=-1)
    structural = plus_ok & eq_ok & digits_ok
    arith_ok = (preds[:, 0] + preds[:, 2]) % dataset.cfg.P == preds[:, 4]
    return {
        "structural_valid": structural.float().mean().item(),
        "arithmetic_correct": (structural & arith_ok).float().mean().item(),
        "first_examples": preds[:5].cpu().tolist(),
    }


def diagnostic_samplers(model, dataset, cfg):
    print("\n=== Diagnostic 3: sampling from uniform noise ===")
    print(f"Vocab: 0-{cfg.P - 1}=digits, {cfg.P}=+, {cfg.P + 1}==")

    for name, fn in [
        ("constant-step  (eta=0.1, 200 steps)", lambda: sample_constant_step(model, cfg, 256, 200, 0.1)),
        ("adaptive-step  (target=0.1, 200 steps)", lambda: sample_adaptive_step(model, cfg, 256, 200)),
        ("temperature-anneal  (1.0 -> 0.05, 200 steps)", lambda: sample_temperature_anneal(model, cfg, 256, 200)),
    ]:
        z = fn()
        scores = decode_and_score(model, dataset, cfg, z)
        print(f"\n  {name}")
        print(f"    structural valid:  {scores['structural_valid']:.3f}")
        print(f"    arithmetic correct: {scores['arithmetic_correct']:.3f}")
        print(f"    first 5 decodes:    {scores['first_examples']}")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------


def main():
    cfg = ToyConfig()
    torch.manual_seed(42)
    model, dataset = train(cfg)
    model.eval()

    diagnostic_energy_at_codewords(model, dataset, cfg)
    diagnostic_gradient_magnitude(model, cfg)
    diagnostic_samplers(model, dataset, cfg)


if __name__ == "__main__":
    main()

/tmp/ipykernel_2957/3260387801.py:185: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.n_layers)


Training on cuda | vocab=9 | d=64
  step   500  loss=0.2143  low-noise acc=1.000
  step  1000  loss=0.1992  low-noise acc=1.000
  step  1500  loss=0.2092  low-noise acc=1.000
  step  2000  loss=0.1757  low-noise acc=1.000
  step  2500  loss=0.1653  low-noise acc=1.000
  step  3000  loss=0.1570  low-noise acc=1.000
  step  3500  loss=0.1432  low-noise acc=1.000
  step  4000  loss=0.1987  low-noise acc=1.000
  step  4500  loss=0.1516  low-noise acc=1.000
  step  5000  loss=0.1688  low-noise acc=1.000
  step  5500  loss=0.1850  low-noise acc=1.000
  step  6000  loss=0.1422  low-noise acc=1.000
  step  6500  loss=0.1598  low-noise acc=1.000
  step  7000  loss=0.1429  low-noise acc=1.000
  step  7500  loss=0.1148  low-noise acc=1.000
  step  8000  loss=0.1840  low-noise acc=1.000

=== Diagnostic 1: energy landscape minima ===
  Mean energy at codewords:    -0.8888  (min: -1.0469)
  Energy at centroid:          -0.9724
  Mean energy at random init:  -0.7329
  Codeword < Centroid:  False  (wa